# Aave V3.1 — data transformation

**Step 1: load & display.** Reads the *latest* versioned CSV per table from
`query_result_data/` and displays it. The raw extracts are treated as
**read-only** — nothing here writes back to that folder.

All loading logic lives in `transform.py` (libraries it imports: `pandas`,
stdlib `re`/`pathlib`, and `data_validation` for the shared query-id → label map).
Edit `TABLE` to choose which table to display; `PREVIEW_ROWS` sets the preview size.

In [126]:
# Cell 1 — imports + notebook-level settings
import pandas as pd
from IPython.display import display
import transform as tf
import data_validation as dv

PREVIEW_ROWS = 10  # rows shown in the inline preview (notebook-only setting)
# Cell 2 — see the available tables (label <- latest file)
for label, filename in sorted(tf.list_tables().items()):
    print(f"{label:30s} <- {filename}")

borrow_repay                   <- query_result_data_7702164_20260612T204729Z.csv
collateral_toggle              <- query_result_data_7711248_20260612T204801Z.csv
flashloan                      <- query_result_data_7711227_20260612T204751Z.csv
liquidation                    <- query_result_data_7711212_20260616T100056Z.csv
oracle_price_usd_eth_weth_6h   <- query_result_data_7714873_20260615T092402Z.csv
query_7711171                  <- query_result_data_7711171_20260612T204720Z.csv
query_7711265                  <- query_result_data_7711265_20260612T204730Z.csv
query_7711276                  <- query_result_data_7711276_20260612T204743Z.csv
query_7711290                  <- query_result_data_7711290_20260612T204747Z.csv
query_7711298                  <- query_result_data_7711298_20260612T204752Z.csv
query_7711304                  <- query_result_data_7711304_20260612T204754Z.csv
query_7711307                  <- query_result_data_7711307_20260612T204802Z.csv
reserve_config              

In [127]:
# Cell 3 — load one table (read-only) and display it
TABLE = "supply_withdraw"  # <-- pick a label from Cell 2 (or pass a CSV path)

df_0_supply_withdraw = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_supply_withdraw.shape[0]} rows x {df_0_supply_withdraw.shape[1]} cols")
display(df_0_supply_withdraw.dtypes.to_frame("dtype"))
display(df_0_supply_withdraw.head(PREVIEW_ROWS))

supply_withdraw: 9684 rows x 12 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
supply_amount_raw,object
withdrawal_amount_raw,object
net_supply_flow_raw,object
supply_tx_count,int64
withdrawal_tx_count,int64
unique_suppliers,int64
unique_withdraw_users,int64


,time_bucket,asset,asset_symbol,supply_amount_raw,withdrawal_amount_raw,net_supply_flow_raw,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,latest_supply_block,latest_withdraw_block
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,NaN,1000000000000000000,-1000000000000000000,0,1,0,1,NaN,23702205.0
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,4799443700,636126131,4163317569,1,4,1,4,23701393.0,23702381.0
2,2025-11-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,436456682654192005,32993234442553860314750,-32992797985871206122745,1,1,1,1,23701338.0,23701353.0
3,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,2141773860,1114070713,1027703147,33,22,23,13,23702517.0,23702526.0
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,206781102780683278444391,288921491161824127738111,-82140388381140849293720,7,4,5,3,23702366.0,23702411.0
5,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,4365431225516880193744,4913534428306924890476,-548103202790044696732,7,3,7,3,23702455.0,23702008.0
6,2025-11-01 00:00:00.000 UTC,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,NaN,439140803371340683542540,-439140803371340683542540,0,5,0,2,NaN,23701785.0
7,2025-11-01 00:00:00.000 UTC,0x62c6e813b9589c3631ba0cdb013acdb8544038b7,PT-USDe-27NOV2025,11500000000000000000000000,NaN,11500000000000000000000000,1,0,1,0,23702251.0,NaN
8,2025-11-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,625719208203236742775418,626859104729841748807474,-1139896526605006032056,5,8,3,6,23701999.0,23701999.0
9,2025-11-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,601228103538,703668780008,-102440676470,8,2,8,2,23702364.0,23700846.0


In [128]:
#7711171
df_1_supply_withdraw = tf.load_table("query_7711171")
print(f"{TABLE}: {df_1_supply_withdraw.shape[0]} rows x {df_1_supply_withdraw.shape[1]} cols")
display(df_1_supply_withdraw.dtypes.to_frame("dtype"))
display(df_1_supply_withdraw.head(PREVIEW_ROWS))
df_0_supply_withdraw.drop(["net_supply_flow_raw", "latest_supply_block", "latest_withdraw_block"], axis=1, inplace=True)
df_1_supply_withdraw.drop(["metric", "unit"], axis=1, inplace=True)
df_1_supply_withdraw = df_1_supply_withdraw.drop(df_1_supply_withdraw.index[[0, 1]])

#oracle_price_usd_eth_weth_6h =  7714873
df_2_supply_withdraw = tf.load_table("oracle_price_usd_eth_weth_6h")
print(f"{TABLE}: {df_2_supply_withdraw.shape[0]} rows x {df_2_supply_withdraw.shape[1]} cols")
display(df_2_supply_withdraw.dtypes.to_frame("dtype"))
display(df_2_supply_withdraw.head(PREVIEW_ROWS))

supply_withdraw: 177 rows x 5 cols


,dtype
asset,str
asset_symbol,str
metric,str
decimals,int64
unit,str


,asset,asset_symbol,metric,decimals,unit
0,NaN,NaN,latest_supply_block,0,block_number
1,NaN,NaN,latest_withdraw_block,0,block_number
2,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,net_supply_flow_raw,18,raw_token_amount
3,0x14bdc3a3ae09f5518b923b69489cbcafb238e617,PT-eUSDE-14AUG2025,net_supply_flow_raw,18,raw_token_amount
4,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,net_supply_flow_raw,18,raw_token_amount
5,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,net_supply_flow_raw,6,raw_token_amount
6,0x1f84a51296691320478c98b8d77f2bbd17d34350,PT-USDe-5FEB2026,net_supply_flow_raw,18,raw_token_amount
7,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,net_supply_flow_raw,18,raw_token_amount
8,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,net_supply_flow_raw,8,raw_token_amount
9,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,net_supply_flow_raw,6,raw_token_amount


supply_withdraw: 17664 rows x 8 cols


,dtype
time_bucket,str
asset,str
symbol,str
decimals,int64
avg_price_usd,float64
avg_price_eth,float64
avg_price_weth,float64
price_points,int64


,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,avg_price_weth,price_points
0,2025-11-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.162976,0.000042,0.000042,6
1,2025-11-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,109757.415833,28.472322,28.472322,6
2,2025-11-01 00:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.155175,0.000300,0.000300,6
3,2025-11-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.775833,0.001498,0.001498,6
4,2025-11-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,109923.351389,28.515371,28.515371,6
5,2025-11-01 00:00:00,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,6,1.099798,0.000285,0.000285,6
6,2025-11-01 00:00:00,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,18,1.000312,0.000259,0.000259,6
7,2025-11-01 00:00:00,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,18,0.999666,0.000259,0.000259,6
8,2025-11-01 00:00:00,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,18,17.231250,0.004470,0.004470,6
9,2025-11-01 00:00:00,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,18,0.872910,0.000226,0.000226,6


In [129]:
# Cell — divide raw amount columns by 10**decimals -> real token units
# decimals come from the reference table (df_1); raw_token_amount rows only.
df_0_supply_withdraw_scaled = tf.scale_by_decimals(df_0_supply_withdraw, df_1_supply_withdraw, drop_raw=True)

print(f"raw cols scaled: {tf.raw_amount_columns(df_0_supply_withdraw)}")
display(df_0_supply_withdraw_scaled.head(PREVIEW_ROWS))
# Cell — validation checks on the scaled amount columns of df_0 (after scale_by_decimals)
import adv_validation as av

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_supply_withdraw)]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_0_supply_withdraw_scaled, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_0_supply_withdraw_scaled, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_0_supply_withdraw_scaled, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")

raw cols scaled: ['supply_amount_raw', 'withdrawal_amount_raw']


,time_bucket,asset,asset_symbol,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount,withdrawal_amount
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,0,1,0,1,NaN,1.000000
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,1,4,1,4,4.799444e+03,636.126131
2,2025-11-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,1,1,1,1,4.364567e-01,32993.234443
3,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,33,22,23,13,2.141774e+01,11.140707
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,7,4,5,3,2.067811e+05,288921.491162
5,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,7,3,7,3,4.365431e+03,4913.534428
6,2025-11-01 00:00:00.000 UTC,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,0,5,0,2,NaN,439140.803371
7,2025-11-01 00:00:00.000 UTC,0x62c6e813b9589c3631ba0cdb013acdb8544038b7,PT-USDe-27NOV2025,1,0,1,0,1.150000e+07,NaN
8,2025-11-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,5,8,3,6,6.257192e+05,626859.104730
9,2025-11-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,8,2,8,2,6.012281e+05,703668.780008


amount columns: ['supply_amount', 'withdrawal_amount']

===== supply_amount =====
negatives : 0 of 7976 (0.0%)

===== withdrawal_amount =====
negatives : 0 of 8157 (0.0%)


In [130]:
df_3_supply_withdraw = tf.multiply_by_price(df_0_supply_withdraw_scaled, df_2_supply_withdraw, amount_cols)
df_3_supply_withdraw

,time_bucket,asset,asset_symbol,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount,withdrawal_amount,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,0,1,0,1,NaN,1.000000e+00,0.000000e+00,0.000000,1.097574e+05,28.472322
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,1,4,1,4,4.799444e+03,6.361261e+02,5.544197e+03,1.438235,7.348370e+02,0.190626
2,2025-11-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,1,1,1,1,4.364567e-01,3.299323e+04,2.520901e+00,0.000654,1.905634e+05,49.434310
3,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,33,22,23,13,2.141774e+01,1.114071e+01,2.354310e+06,610.734767,1.224624e+06,317.681400
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,7,4,5,3,2.067811e+05,2.889215e+05,2.067121e+05,53.623732,2.888251e+05,74.924877
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9679,2026-01-31 18:00:00.000 UTC,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,2,4,2,4,1.224273e+06,1.856667e+05,1.224848e+06,510.286479,1.857538e+05,77.387322
9680,2026-01-31 18:00:00.000 UTC,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,1,7,1,5,6.000000e+03,3.824900e+06,6.003044e+03,2.500943,3.826841e+06,1594.309510
9681,2026-01-31 18:00:00.000 UTC,0xe8483517077afa11a9b07f849cee2552f040d7b2,PT-sUSDE-5FEB2026,0,5,0,1,NaN,2.208903e+06,0.000000e+00,0.000000,0.000000e+00,0.000000
9682,2026-01-31 18:00:00.000 UTC,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,osETH,1,0,1,0,3.052752e+01,NaN,7.791735e+04,32.458888,0.000000e+00,0.000000


In [131]:
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]

df_4_supply_withdraw = tf.aggregate_by_time_bucket(df_3_supply_withdraw, "time_bucket", amount_cols, agg_func='sum')
df_4_supply_withdraw

,time_bucket,supply_tx_count,withdrawal_tx_count,unique_suppliers,unique_withdraw_users,supply_amount_value_usd,supply_amount_value_eth,withdrawal_amount_value_usd,withdrawal_amount_value_eth
0,2025-11-01 00:00:00.000 UTC,317,305,191,197,5.817267e+08,150906.828593,7.058171e+08,183097.265312
1,2025-11-01 06:00:00.000 UTC,391,291,213,171,8.982494e+07,23209.831528,8.199260e+07,21186.012501
2,2025-11-01 12:00:00.000 UTC,425,307,205,153,7.963634e+07,20522.630436,9.197376e+07,23702.000478
3,2025-11-01 18:00:00.000 UTC,304,181,171,107,1.781438e+08,45918.055714,1.730435e+08,44603.394329
4,2025-11-02 00:00:00.000 UTC,252,177,144,96,8.335144e+07,21467.127778,5.078573e+07,13079.799820
...,...,...,...,...,...,...,...,...,...
363,2026-01-30 18:00:00.000 UTC,896,795,459,433,2.801822e+08,103558.751670,3.599294e+08,133034.891187
364,2026-01-31 00:00:00.000 UTC,473,385,275,208,2.098444e+08,77742.495749,1.368650e+08,50705.303967
365,2026-01-31 06:00:00.000 UTC,717,702,374,369,1.835683e+08,68996.377735,1.623178e+08,61009.026540
366,2026-01-31 12:00:00.000 UTC,1910,1900,826,1037,4.868281e+08,190597.093328,6.604470e+08,258595.845521


In [132]:
# Cell — validation checks on the scaled amount columns of df_0 (after scale_by_decimals)
import adv_validation as av

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [
    "supply_tx_count",
    "withdrawal_tx_count",
    "unique_suppliers",
    "unique_withdraw_users",
    "supply_amount_value_usd",
    "supply_amount_value_eth",
    "withdrawal_amount_value_usd",
    "withdrawal_amount_value_eth",
]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")


    neg = av.negative_value_check(df_4_supply_withdraw, col, plot=False)
    # rng = av.range_check(df_4_supply_withdraw, col, plot=False)
    # dev = av.deviation_score(df_4_supply_withdraw, col, plot=False)

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")


amount columns: ['supply_tx_count', 'withdrawal_tx_count', 'unique_suppliers', 'unique_withdraw_users', 'supply_amount_value_usd', 'supply_amount_value_eth', 'withdrawal_amount_value_usd', 'withdrawal_amount_value_eth']

===== supply_tx_count =====
negatives : 0 of 368 (0.0%)

===== withdrawal_tx_count =====
negatives : 0 of 368 (0.0%)

===== unique_suppliers =====
negatives : 0 of 368 (0.0%)

===== unique_withdraw_users =====
negatives : 0 of 368 (0.0%)

===== supply_amount_value_usd =====
negatives : 0 of 368 (0.0%)

===== supply_amount_value_eth =====
negatives : 0 of 368 (0.0%)

===== withdrawal_amount_value_usd =====
negatives : 0 of 368 (0.0%)

===== withdrawal_amount_value_eth =====
negatives : 0 of 368 (0.0%)


In [133]:
stats = pd.DataFrame({
    "mean": df_4_supply_withdraw[amount_cols].mean(),
    "median": df_4_supply_withdraw[amount_cols].median(),
    "std": df_4_supply_withdraw[amount_cols].std(),
    "variance": df_4_supply_withdraw[amount_cols].var(),
    "cv": df_4_supply_withdraw[amount_cols].std() / df_4_supply_withdraw[amount_cols].mean(),
    "p95": df_4_supply_withdraw[amount_cols].quantile(0.95),
    "p99": df_4_supply_withdraw[amount_cols].quantile(0.99)
})

print(stats)

                                     mean        median           std  \
supply_tx_count              5.649293e+02  5.010000e+02  2.708089e+02   
withdrawal_tx_count          4.949565e+02  4.390000e+02  2.494681e+02   
unique_suppliers             2.947908e+02  2.705000e+02  1.189880e+02   
unique_withdraw_users        2.603696e+02  2.400000e+02  1.068772e+02   
supply_amount_value_usd      2.554288e+08  1.977666e+08  2.295273e+08   
supply_amount_value_eth      8.259261e+04  6.488047e+04  7.506401e+04   
withdrawal_amount_value_usd  2.385869e+08  1.630199e+08  2.830554e+08   
withdrawal_amount_value_eth  7.718386e+04  5.277488e+04  9.216219e+04   

                                 variance        cv           p95  \
supply_tx_count              7.333747e+04  0.479368  9.926500e+02   
withdrawal_tx_count          6.223434e+04  0.504020  9.085000e+02   
unique_suppliers             1.415813e+04  0.403635  4.979000e+02   
unique_withdraw_users        1.142275e+04  0.410483  4.343000e+02 

In [134]:
# load one table (read-only) and display it
TABLE = "borrow_repay"  # <-- pick a label from Cell 2 (or pass a CSV path)

df_0_borrow = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_borrow.shape[0]} rows x {df_0_borrow.shape[1]} cols")
display(df_0_borrow.dtypes.to_frame("dtype"))
display(df_0_borrow.head(PREVIEW_ROWS))

borrow_repay: 6164 rows x 15 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
borrow_amount_raw,object
repay_amount_raw,object
net_debt_flow_raw,object
borrow_tx_count,int64
repay_tx_count,int64
stable_borrow_tx_count,int64
variable_borrow_tx_count,int64


,time_bucket,asset,asset_symbol,borrow_amount_raw,repay_amount_raw,net_debt_flow_raw,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,last_borrow_rate,latest_borrow_block,latest_repay_block
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,1000000000000000000,NaN,1000000000000000000,1,0,0,1,1,0,3289582737764284284996091,23702433.0,NaN
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,NaN,75583025,-75583025,0,1,0,0,0,1,NaN,NaN,23702174.0
2,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,11116289339,11237510413,-121221074,6,12,0,6,3,8,2061326832260803817283790,23701822.0,23702109.0
3,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,611989843191524873525,342383985642956817940378,-341771995799765293066853,3,4,0,3,3,4,52500000000000000000000000,23702446.0,23702160.0
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,800000000000000000000000,30723511805762261164197771,-29923511805762261164197771,3,14,0,3,1,11,47932673283857703530178210,23702076.0,23702511.0
5,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,108864699718269860777,519448362627564363339,-410583662909294502562,4,5,0,4,4,4,5480377192893860444224220,23701797.0,23701282.0
6,2025-11-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,1032701624199906192486858,1129556620776557449381311,-96854996576651256894453,4,4,0,4,3,3,55724099349395478242715330,23701992.0,23701467.0
7,2025-11-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,26109089895969,244580695577,25864509200392,18,5,0,18,9,3,26825850433942179209871059,23702472.0,23702393.0
8,2025-11-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,10000000000000000000,21200031159693236910,-11200031159693236910,1,2,0,1,1,2,1950859333740656407395979,23702146.0,23702091.0
9,2025-11-01 00:00:00.000 UTC,0x8292bb45bf1ee4d140127049757c2e0ff06317ed,RLUSD,196663000000000000000000,1400979257857249476347285,-1204316257857249476347285,2,3,0,2,1,2,42601953385032056602639726,23700844.0,23701970.0


In [135]:
# df_1 = tf.load_table("query_7711265")
# print(f"{TABLE}: {df_1.shape[0]} rows x {df_1.shape[1]} cols")
# display(df_1.dtypes.to_frame("dtype"))
# display(df_1.head(PREVIEW_ROWS))

df_0_borrow["last_borrow_rate"] = df_0_borrow["last_borrow_rate"].astype(float) / 10**27
df_0_borrow

,time_bucket,asset,asset_symbol,borrow_amount_raw,repay_amount_raw,net_debt_flow_raw,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,last_borrow_rate,latest_borrow_block,latest_repay_block
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,1000000000000000000,NaN,1000000000000000000,1,0,0,1,1,0,0.003290,23702433.0,NaN
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,NaN,75583025,-75583025,0,1,0,0,0,1,NaN,NaN,23702174.0
2,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,11116289339,11237510413,-121221074,6,12,0,6,3,8,0.002061,23701822.0,23702109.0
3,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,611989843191524873525,342383985642956817940378,-341771995799765293066853,3,4,0,3,3,4,0.052500,23702446.0,23702160.0
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,800000000000000000000000,30723511805762261164197771,-29923511805762261164197771,3,14,0,3,1,11,0.047933,23702076.0,23702511.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6159,2026-01-31 18:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,7951693782,2750923358,5200770424,17,13,0,17,7,10,0.003899,24358044.0,24358024.0
6160,2026-01-31 18:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,98636925892414,76089230465070,22547695427344,318,602,0,318,225,441,0.039752,24358283.0,24358291.0
6161,2026-01-31 18:00:00.000 UTC,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,NaN,72491960175839071714968,-72491960175839071714968,0,1,0,0,0,1,NaN,NaN,24356900.0
6162,2026-01-31 18:00:00.000 UTC,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6000000000,6716206082701,-6710206082701,1,5,0,1,1,5,0.024720,24357854.0,24357765.0


In [136]:
#oracle_price_usd_eth_weth_6h =  7714873
df_1_borrow = tf.load_table("oracle_price_usd_eth_weth_6h")
print(f"{TABLE}: {df_1_borrow.shape[0]} rows x {df_1_borrow.shape[1]} cols")
display(df_1_borrow.dtypes.to_frame("dtype"))
display(df_1_borrow.head(PREVIEW_ROWS))

borrow_repay: 17664 rows x 8 cols


,dtype
time_bucket,str
asset,str
symbol,str
decimals,int64
avg_price_usd,float64
avg_price_eth,float64
avg_price_weth,float64
price_points,int64


,time_bucket,asset,symbol,decimals,avg_price_usd,avg_price_eth,avg_price_weth,price_points
0,2025-11-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18,0.162976,0.000042,0.000042,6
1,2025-11-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,18,109757.415833,28.472322,28.472322,6
2,2025-11-01 00:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,6,1.155175,0.000300,0.000300,6
3,2025-11-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,18,5.775833,0.001498,0.001498,6
4,2025-11-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,8,109923.351389,28.515371,28.515371,6
5,2025-11-01 00:00:00,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,6,1.099798,0.000285,0.000285,6
6,2025-11-01 00:00:00,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,18,1.000312,0.000259,0.000259,6
7,2025-11-01 00:00:00,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,18,0.999666,0.000259,0.000259,6
8,2025-11-01 00:00:00,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,18,17.231250,0.004470,0.004470,6
9,2025-11-01 00:00:00,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,18,0.872910,0.000226,0.000226,6


In [137]:
df_0_borrow.drop(["net_debt_flow_raw","latest_borrow_block", "latest_repay_block"], axis=1, inplace=True)
DF_common = df_0_borrow[["time_bucket", "asset", "asset_symbol", "last_borrow_rate"]].copy()
df_0_borrow.drop(["last_borrow_rate"], axis=1, inplace=True)
df_0_borrow

,time_bucket,asset,asset_symbol,borrow_amount_raw,repay_amount_raw,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,1000000000000000000,NaN,1,0,0,1,1,0
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,NaN,75583025,0,1,0,0,0,1
2,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,11116289339,11237510413,6,12,0,6,3,8
3,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,611989843191524873525,342383985642956817940378,3,4,0,3,3,4
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,800000000000000000000000,30723511805762261164197771,3,14,0,3,1,11
...,...,...,...,...,...,...,...,...,...,...,...
6159,2026-01-31 18:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,7951693782,2750923358,17,13,0,17,7,10
6160,2026-01-31 18:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,98636925892414,76089230465070,318,602,0,318,225,441
6161,2026-01-31 18:00:00.000 UTC,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,NaN,72491960175839071714968,0,1,0,0,0,1
6162,2026-01-31 18:00:00.000 UTC,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,6000000000,6716206082701,1,5,0,1,1,5


In [138]:
# Cell — divide borrow_repay raw amount columns by 10**decimals -> real token units
# decimals come from df_1_borrow (the oracle_price table's `decimals` column), per asset.
df_2_borrow = tf.scale_by_decimals(df_0_borrow, df_1_borrow, drop_raw=True)

print(f"raw cols scaled: {tf.raw_amount_columns(df_0_borrow)}")
display(df_2_borrow.head(PREVIEW_ROWS))

# the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_borrow)]
print("amount columns:", amount_cols)

# for each amount column: negative-value check, range check, deviation score
# (plot=False -> values only, no plots)
for col in amount_cols:
    print(f"\n===== {col} =====")
    neg = av.negative_value_check(df_2_borrow, col, plot=False)   # any negatives? how many?
    # rng = av.range_check(df_2_borrow, col, plot=False)            # observed value range
    # dev = av.deviation_score(df_2_borrow, col, plot=False)        # per-row score in [-1, 1]

    print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
    # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
    # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")



raw cols scaled: ['borrow_amount_raw', 'repay_amount_raw']


,time_bucket,asset,asset_symbol,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,borrow_amount,repay_amount
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,1,0,0,1,1,0,1.000000e+00,NaN
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,0,1,0,0,0,1,NaN,7.558303e+01
2,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,6,12,0,6,3,8,1.111629e+02,1.123751e+02
3,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,3,4,0,3,3,4,6.119898e+02,3.423840e+05
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,3,14,0,3,1,11,8.000000e+05,3.072351e+07
5,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,4,5,0,4,4,4,1.088647e+02,5.194484e+02
6,2025-11-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,4,4,0,4,3,3,1.032702e+06,1.129557e+06
7,2025-11-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,18,5,0,18,9,3,2.610909e+07,2.445807e+05
8,2025-11-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,1,2,0,1,1,2,1.000000e+01,2.120003e+01
9,2025-11-01 00:00:00.000 UTC,0x8292bb45bf1ee4d140127049757c2e0ff06317ed,RLUSD,2,3,0,2,1,2,1.966630e+05,1.400979e+06


amount columns: ['borrow_amount', 'repay_amount']

===== borrow_amount =====
negatives : 0 of 5446 (0.0%)

===== repay_amount =====
negatives : 0 of 5544 (0.0%)


In [139]:


df_2_borrow = tf.multiply_by_price(df_2_borrow, df_1_borrow, amount_cols)
df_2_borrow



,time_bucket,asset,asset_symbol,borrow_tx_count,repay_tx_count,stable_borrow_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,borrow_amount,repay_amount,borrow_amount_value_usd,borrow_amount_value_eth,repay_amount_value_usd,repay_amount_value_eth
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,1,0,0,1,1,0,1.000000e+00,NaN,1.097574e+05,28.472322,0.000000e+00,0.000000
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,0,1,0,0,0,1,NaN,7.558303e+01,0.000000e+00,0.000000,8.731162e+01,0.022650
2,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,6,12,0,6,3,8,1.111629e+02,1.123751e+02,1.221940e+07,3169.851174,1.235265e+07,3204.417813
3,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,3,4,0,3,3,4,6.119898e+02,3.423840e+05,6.121810e+02,0.158807,3.424909e+05,88.846471
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,3,14,0,3,1,11,8.000000e+05,3.072351e+07,7.997331e+05,207.460861,3.071326e+07,7967.407780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6159,2026-01-31 18:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,17,13,0,17,7,10,7.951694e+01,2.750923e+01,6.204269e+06,2584.691547,2.146394e+06,894.185383
6160,2026-01-31 18:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,318,602,0,318,225,441,9.863693e+07,7.608923e+07,9.856349e+07,41062.757892,7.603258e+07,31676.105277
6161,2026-01-31 18:00:00.000 UTC,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,0,1,0,0,0,1,NaN,7.249196e+04,0.000000e+00,0.000000,7.252599e+04,30.215217
6162,2026-01-31 18:00:00.000 UTC,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,1,5,0,1,1,5,6.000000e+03,6.716206e+06,6.003044e+03,2.500943,6.719613e+06,2799.474829


In [140]:
borrow_agg_cols = [
    "borrow_tx_count",
    "repay_tx_count",
    # "stable_borrow_tx_count",
    "variable_borrow_tx_count",
    "unique_borrowers",
    "unique_repayers",
    "borrow_amount_value_usd",
    "borrow_amount_value_eth",
    "repay_amount_value_usd",
    "repay_amount_value_eth",
]

df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')
# df_3_borrow.drop(["stable_borrow_tx_count"], axis=1, inplace=True )

df_3_borrow

,time_bucket,borrow_tx_count,repay_tx_count,variable_borrow_tx_count,unique_borrowers,unique_repayers,borrow_amount_value_usd,borrow_amount_value_eth,repay_amount_value_usd,repay_amount_value_eth
0,2025-11-01 00:00:00.000 UTC,197,162,197,140,106,2.768394e+08,71815.616832,2.542997e+08,65968.511196
1,2025-11-01 06:00:00.000 UTC,185,140,185,137,88,3.437619e+07,8882.465331,3.039276e+07,7853.186734
2,2025-11-01 12:00:00.000 UTC,286,219,286,180,120,4.453172e+07,11476.032964,4.300053e+07,11081.435598
3,2025-11-01 18:00:00.000 UTC,186,161,186,130,82,3.679280e+07,9483.662321,2.502057e+07,6449.265968
4,2025-11-02 00:00:00.000 UTC,157,129,157,101,63,5.766589e+07,14851.848163,3.190852e+07,8218.009991
...,...,...,...,...,...,...,...,...,...,...
363,2026-01-30 18:00:00.000 UTC,438,510,438,250,302,1.335236e+08,49352.209379,2.362376e+08,87317.232958
364,2026-01-31 00:00:00.000 UTC,189,161,189,120,113,7.131898e+07,26422.041669,7.885991e+07,29215.788021
365,2026-01-31 06:00:00.000 UTC,356,423,356,246,286,3.809374e+07,14318.207007,5.541483e+07,20828.663051
366,2026-01-31 12:00:00.000 UTC,602,1562,602,396,1074,2.006427e+08,78576.917823,3.774539e+08,147852.612718


In [141]:
stats = pd.DataFrame({
    "mean": df_3_borrow[borrow_agg_cols].mean(),
    "median": df_3_borrow[borrow_agg_cols].median(),
    "std": df_3_borrow[borrow_agg_cols].std(),
    "variance": df_3_borrow[borrow_agg_cols].var(),
    "cv": df_3_borrow[borrow_agg_cols].std() / df_3_borrow[borrow_agg_cols].mean(),
    "p95": df_3_borrow[borrow_agg_cols].quantile(0.95),
    "p99": df_3_borrow[borrow_agg_cols].quantile(0.99)
})

print(stats)

                                  mean        median           std  \
borrow_tx_count           2.619755e+02  2.380000e+02  1.105615e+02   
repay_tx_count            2.561223e+02  1.950000e+02  2.200325e+02   
variable_borrow_tx_count  2.619755e+02  2.380000e+02  1.105615e+02   
unique_borrowers          1.737527e+02  1.590000e+02  6.617857e+01   
unique_repayers           1.513152e+02  1.200000e+02  1.180763e+02   
borrow_amount_value_usd   8.688762e+07  7.112369e+07  6.959942e+07   
borrow_amount_value_eth   2.805171e+04  2.299531e+04  2.230847e+04   
repay_amount_value_usd    8.213836e+07  6.287839e+07  7.324224e+07   
repay_amount_value_eth    2.657937e+04  2.006347e+04  2.377873e+04   

                              variance        cv           p95           p99  
borrow_tx_count           1.222384e+04  0.422030  4.616000e+02  5.749400e+02  
repay_tx_count            4.841431e+04  0.859092  6.308000e+02  1.202880e+03  
variable_borrow_tx_count  1.222384e+04  0.422030  4.616000e+02

In [142]:
TABLE = "reserve_state_rates"  

df_0_reserve_state_rates = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_reserve_state_rates.shape[0]} rows x {df_0_reserve_state_rates.shape[1]} cols")
display(df_0_reserve_state_rates.dtypes.to_frame("dtype"))
display(df_0_reserve_state_rates.head(PREVIEW_ROWS))

reserve_state_rates: 10907 rows x 8 cols


,dtype
time_bucket,str
asset,str
liquidity_rate,object
variable_borrow_rate,object
stable_borrow_rate,int64
liquidity_index,object
variable_borrow_index,object
update_count,int64


,time_bucket,asset,liquidity_rate,variable_borrow_rate,stable_borrow_rate,liquidity_index,variable_borrow_index,update_count
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,108213544359955268078222,3289582737764284284996091,0,1000123425008273445786612988,1002229352148787934859365850,2
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,38051955062670229951344701,53091085296644413096761806,0,1012935214596446036418236814,1018158686348867753240265398,6
2,2025-11-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,2827703180154487102271862,23448484834888865255329369,0,1002977614298512869530506162,1025316407837083198922446666,2
3,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,37595904960613720881440,1938966378355600772812388,0,1003386230030616482557914284,1024056502484912237004803258,74
4,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,0,52500000000000000000000000,0,1000000000000000000000000000,1159328274706921345182327089,7
5,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,24595051033522578123775920,47721269493808002008013088,0,1056670140577291734449123015,1108488290631474184970130336,29
6,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,154407935009196093665724,5479395217051567776356864,0,1000810852120167999495087238,1010175408092432187510346271,19
7,2025-11-01 00:00:00.000 UTC,0x5a98fcbea516cf06857215779fd812ca3bef1b32,1757784356887471249611184,18487601324350311985949236,0,1005324779722595567653717476,1030492042490140917344514157,5
8,2025-11-01 00:00:00.000 UTC,0x62c6e813b9589c3631ba0cdb013acdb8544038b7,0,0,0,1000000000000000000000000000,1000000000000000000000000000,1
9,2025-11-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,35722721057069715073784428,55734858505050390880307704,0,1142630661646218620115001384,1215562293061344379259328502,21


In [143]:
reserve_agg_cols = [
    "liquidity_rate",
    "stable_borrow_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

for col in reserve_agg_cols:
    df_0_reserve_state_rates[col] = df_0_reserve_state_rates[col].astype(float) / 10**27

df_0_reserve_state_rates

,time_bucket,asset,liquidity_rate,variable_borrow_rate,stable_borrow_rate,liquidity_index,variable_borrow_index,update_count
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,1.082135e-04,3.289583e-03,0.0,1.000123,1.002229,2
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,3.805196e-02,5.309109e-02,0.0,1.012935,1.018159,6
2,2025-11-01 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,2.827703e-03,2.344848e-02,0.0,1.002978,1.025316,2
3,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,3.759590e-05,1.938966e-03,0.0,1.003386,1.024057,74
4,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,0.000000e+00,5.250000e-02,0.0,1.000000,1.159328,7
...,...,...,...,...,...,...,...,...
10902,2026-01-31 18:00:00.000 UTC,0xdc035d45d973e3ec169d2276ddab16f1e407384f,5.373868e-03,5.604228e-02,0.0,1.037697,1.094960,10
10903,2026-01-31 18:00:00.000 UTC,0xe343167631d89b6ffc58b88d6b7fb0228795491d,6.515127e-03,2.471423e-02,0.0,1.001421,1.003561,14
10904,2026-01-31 18:00:00.000 UTC,0xe8483517077afa11a9b07f849cee2552f040d7b2,0.000000e+00,0.000000e+00,0.0,1.000000,1.000000,5
10905,2026-01-31 18:00:00.000 UTC,0xf1c9acdc66974dfb6decb12aa385b9cd01190e38,2.892040e-12,7.275042e-07,0.0,1.000145,1.002955,1


In [144]:
DF_common_1 = DF_common.merge(
    df_0_reserve_state_rates,
      on=["time_bucket", "asset"], how="left", suffixes=("_x", "_y"))

DF_common_1.drop(["stable_borrow_rate"], axis=1, inplace=True)
DF_common_1

,time_bucket,asset,asset_symbol,last_borrow_rate,liquidity_rate,variable_borrow_rate,liquidity_index,variable_borrow_index,update_count
0,2025-11-01 00:00:00.000 UTC,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,0.003290,0.000108,0.003290,1.000123,1.002229,2
1,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,NaN,0.038052,0.053091,1.012935,1.018159,6
2,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,0.002061,0.000038,0.001939,1.003386,1.024057,74
3,2025-11-01 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0.052500,0.000000,0.052500,1.000000,1.159328,7
4,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,0.047933,0.024595,0.047721,1.056670,1.108488,29
...,...,...,...,...,...,...,...,...,...
6159,2026-01-31 18:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,0.003899,0.000055,0.003899,1.000191,1.003510,160
6160,2026-01-31 18:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,0.039752,0.026169,0.039752,1.154369,1.214214,1845
6161,2026-01-31 18:00:00.000 UTC,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,NaN,0.005374,0.056042,1.037697,1.094960,10
6162,2026-01-31 18:00:00.000 UTC,0xe343167631d89b6ffc58b88d6b7fb0228795491d,USDG,0.024720,0.006515,0.024714,1.001421,1.003561,14


In [145]:
common_cols = [
    "last_borrow_rate",
    "liquidity_rate",
    "variable_borrow_rate",
    "liquidity_index",
    "variable_borrow_index",
]

stats = pd.DataFrame({
    "mean": DF_common_1[common_cols].mean(),
    "median": DF_common_1[common_cols].median(),
    "std": DF_common_1[common_cols].std(),
    "variance": DF_common_1[common_cols].var(),
    "cv": DF_common_1[common_cols].std() / DF_common_1[common_cols].mean(),
    "p95": DF_common_1[common_cols].quantile(0.95),
    "p99": DF_common_1[common_cols].quantile(0.99)
})

print(stats)

                           mean    median       std  variance        cv  \
last_borrow_rate       0.031700  0.035716  0.036699  0.001347  1.157715   
liquidity_rate         0.013370  0.012011  0.015411  0.000238  1.152653   
variable_borrow_rate   0.029220  0.035464  0.022352  0.000500  0.764966   
liquidity_index        1.048845  1.012431  0.061065  0.003729  0.058221   
variable_borrow_index  1.090588  1.032202  0.085383  0.007290  0.078290   

                            p95       p99  
last_borrow_rate       0.056707  0.080597  
liquidity_rate         0.037120  0.046863  
variable_borrow_rate   0.056653  0.061219  
liquidity_index        1.158282  1.179738  
variable_borrow_index  1.227677  1.249422  


In [146]:
TABLE = "reserve_config" 

df_0_reserve_config = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_reserve_config.shape[0]} rows x {df_0_reserve_config.shape[1]} cols")
display(df_0_reserve_config.dtypes.to_frame("dtype"))
display(df_0_reserve_config.head(PREVIEW_ROWS))

reserve_config: 21434 rows x 11 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
supply_cap,int64
borrow_cap,int64
debt_ceiling,int64
reserve_factor,int64
liquidation_threshold,int64
liquidation_bonus,int64
ltv,int64


,time_bucket,asset,asset_symbol,supply_cap,borrow_cap,debt_ceiling,reserve_factor,liquidation_threshold,liquidation_bonus,ltv,config_event_count
0,2025-11-01 00:00:00,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,18000000,1,117964800,2000,6700,10750,5700,0
1,2025-11-01 00:00:00,0x14bdc3a3ae09f5518b923b69489cbcafb238e617,PT-eUSDE-14AUG2025,1,1,0,4500,10,10750,5,0
2,2025-11-01 00:00:00,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,2600,275,0,5000,7800,10750,7300,0
3,2025-11-01 00:00:00,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,70000000,64000000,0,1000,7800,10500,7500,0
4,2025-11-01 00:00:00,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,6000000,1,1700000000,2000,7400,11000,6500,0
5,2025-11-01 00:00:00,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,49875,28000,0,5000,7800,10500,7300,0
6,2025-11-01 00:00:00,0x3432b6a60d23ca0dfca7761b7ab56459d9c964d0,FXS,1200000,330000,400000000,2000,4200,11000,0,0
7,2025-11-01 00:00:00,0x3b3fb9c57858ef816833dc91565efcd85d96f634,PT-sUSDE-31JUL2025,1,1,0,2000,10,10750,5,0
8,2025-11-01 00:00:00,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,1,180000000,0,10000,0,0,0,0
9,2025-11-01 00:00:00,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,2700000000,2500000000,0,2500,7500,10850,7200,0


In [147]:
summary = pd.DataFrame({
    "null_count": df_0_reserve_config.isna().sum(),
    "null_pct": df_0_reserve_config.isna().mean() * 100,
})
print(summary)

# reserve config results is broken, not enough nor consistent data with very high percentage of null values, so we will not use it for now.

                       null_count  null_pct
time_bucket                     0       0.0
asset                           0       0.0
asset_symbol                    0       0.0
supply_cap                      0       0.0
borrow_cap                      0       0.0
debt_ceiling                    0       0.0
reserve_factor                  0       0.0
liquidation_threshold           0       0.0
liquidation_bonus               0       0.0
ltv                             0       0.0
config_event_count              0       0.0


In [148]:
TABLE = "query_7711290"

df_1_reserve_config = tf.load_table(TABLE)
print(f"{TABLE}: {df_1_reserve_config.shape[0]} rows x {df_1_reserve_config.shape[1]} cols")
display(df_1_reserve_config.dtypes.to_frame("dtype"))
display(df_1_reserve_config.head(PREVIEW_ROWS))

query_7711290: 82 rows x 5 cols


,dtype
asset,str
asset_symbol,str
metric,str
decimals,int64
unit,str


,asset,asset_symbol,metric,decimals,unit
0,NaN,NaN,borrow_cap,0,whole_tokens
1,NaN,NaN,config_event_count,0,count
2,NaN,NaN,debt_ceiling,2,usd_2dp
3,NaN,NaN,latest_liquidation_block,0,block_number
4,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,liquidated_collateral_raw,18,raw_token_amount
5,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,liquidated_collateral_raw,18,raw_token_amount
6,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,liquidated_collateral_raw,18,raw_token_amount
7,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,liquidated_collateral_raw,8,raw_token_amount
8,0x3b3fb9c57858ef816833dc91565efcd85d96f634,PT-sUSDE-31JUL2025,liquidated_collateral_raw,18,raw_token_amount
9,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,liquidated_collateral_raw,18,raw_token_amount


In [149]:
TABLE = "liquidation" 

df_0_liquidation = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_liquidation.shape[0]} rows x {df_0_liquidation.shape[1]} cols")
display(df_0_liquidation.dtypes.to_frame("dtype"))
# display(df_0_liquidation.head(PREVIEW_ROWS))
df_0_liquidation

liquidation: 1006 rows x 12 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
liquidated_collateral_raw,object
liquidation_debt_covered_raw,object
as_collateral_tx_count,int64
as_debt_tx_count,int64
liquidation_tx_count,int64
receive_atoken_count,int64
unique_liquidated_users,int64


,time_bucket,asset,asset_symbol,liquidated_collateral_raw,liquidation_debt_covered_raw,as_collateral_tx_count,as_debt_tx_count,liquidation_tx_count,receive_atoken_count,unique_liquidated_users,unique_liquidators,latest_liquidation_block
0,2025-11-03 00:00:00.000 UTC,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,67491474673428908526,0,1,0,1,0,1,1,23716127
1,2025-11-03 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,2340549,1983150,1,1,2,0,2,2,23716283
2,2025-11-03 00:00:00.000 UTC,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,0,40272071620790000000000,0,1,1,0,1,1,23716283
3,2025-11-03 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,138777763193529721490,0,1,0,1,0,1,1,23716127
4,2025-11-03 00:00:00.000 UTC,0x657e8c867d8b37dcc18fa4caead9c45eb088c642,eBTC,154083,0,1,0,1,0,1,1,23715836
...,...,...,...,...,...,...,...,...,...,...,...,...
1001,2026-01-31 18:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,839249272,0,10,0,10,0,10,6,24356743
1002,2026-01-31 18:00:00.000 UTC,0xcd5fe23c85820f7b72d0926fc9b05b43e359b7ee,weETH,114823957539984722480,0,6,0,6,0,6,4,24357955
1003,2026-01-31 18:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,15472212294,51616235055678,5,205,210,0,182,21,24358017
1004,2026-01-31 18:00:00.000 UTC,0xdc035d45d973e3ec169d2276ddab16f1e407384f,USDS,0,123616494357987029170041,0,2,2,0,2,2,24356696


In [150]:
TABLE = "query_7711290" 

df_0_liq_ = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_liq_.shape[0]} rows x {df_0_liq_.shape[1]} cols")
display(df_0_liq_.dtypes.to_frame("dtype"))
display(df_0_liq_.head(PREVIEW_ROWS))

query_7711290: 82 rows x 5 cols


,dtype
asset,str
asset_symbol,str
metric,str
decimals,int64
unit,str


,asset,asset_symbol,metric,decimals,unit
0,NaN,NaN,borrow_cap,0,whole_tokens
1,NaN,NaN,config_event_count,0,count
2,NaN,NaN,debt_ceiling,2,usd_2dp
3,NaN,NaN,latest_liquidation_block,0,block_number
4,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,liquidated_collateral_raw,18,raw_token_amount
5,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,liquidated_collateral_raw,18,raw_token_amount
6,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,liquidated_collateral_raw,18,raw_token_amount
7,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,liquidated_collateral_raw,8,raw_token_amount
8,0x3b3fb9c57858ef816833dc91565efcd85d96f634,PT-sUSDE-31JUL2025,liquidated_collateral_raw,18,raw_token_amount
9,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,liquidated_collateral_raw,18,raw_token_amount


In [151]:
TABLE = "collateral_toggle"

df_0_collateral = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_collateral.shape[0]} rows x {df_0_collateral.shape[1]} cols")
display(df_0_collateral.dtypes.to_frame("dtype"))
display(df_0_collateral.head(PREVIEW_ROWS))

collateral_toggle: 7019 rows x 8 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
collateral_enabled_count,int64
collateral_disabled_count,int64
unique_collateral_enable_users,int64
unique_collateral_disable_users,int64
latest_collateral_toggle_block,int64


,time_bucket,asset,asset_symbol,collateral_enabled_count,collateral_disabled_count,unique_collateral_enable_users,unique_collateral_disable_users,latest_collateral_toggle_block
0,2025-11-01 00:00:00.000 UTC,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,2,2,2,2,23701380
1,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,23,19,16,13,23702526
2,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,2,2,2,2,23701499
3,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,3,1,3,1,23702455
4,2025-11-01 00:00:00.000 UTC,0x5a98fcbea516cf06857215779fd812ca3bef1b32,LDO,0,1,0,1,23701785
5,2025-11-01 00:00:00.000 UTC,0x6b175474e89094c44da98b954eedeac495271d0f,DAI,5,4,3,2,23701999
6,2025-11-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,7,2,7,2,23702364
7,2025-11-01 00:00:00.000 UTC,0x7f39c581f595b53c5cb19bd0b3f8da6c935e2ca0,wstETH,7,7,6,6,23702555
8,2025-11-01 00:00:00.000 UTC,0x7fc66500c84a76ad7e9c93437bfc5ac33e2ddae9,AAVE,2,1,2,1,23702027
9,2025-11-01 00:00:00.000 UTC,0x8236a87084f8b84306f72007f36f2618a5634494,LBTC,0,1,0,1,23700794


In [152]:
collateral_agg_cols = [
    "collateral_enabled_count",
    "collateral_disabled_count",
    "unique_collateral_enable_users",
    "unique_collateral_disable_users"
]

df_1_collateral = tf.aggregate_by_time_bucket(df_0_collateral, "time_bucket", collateral_agg_cols, agg_func='sum')
df_1_collateral

,time_bucket,collateral_enabled_count,collateral_disabled_count,unique_collateral_enable_users,unique_collateral_disable_users
0,2025-11-01 00:00:00.000 UTC,208,178,159,129
1,2025-11-01 06:00:00.000 UTC,241,176,186,122
2,2025-11-01 12:00:00.000 UTC,267,212,177,122
3,2025-11-01 18:00:00.000 UTC,222,144,166,89
4,2025-11-02 00:00:00.000 UTC,163,119,119,74
...,...,...,...,...,...
363,2026-01-30 18:00:00.000 UTC,224,207,175,160
364,2026-01-31 00:00:00.000 UTC,155,115,129,90
365,2026-01-31 06:00:00.000 UTC,209,198,182,173
366,2026-01-31 12:00:00.000 UTC,396,431,334,370


In [153]:
TABLE = "flashloan"

df_0_flashloan = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_flashloan.shape[0]} rows x {df_0_flashloan.shape[1]} cols")
display(df_0_flashloan.dtypes.to_frame("dtype"))
display(df_0_flashloan.head(PREVIEW_ROWS))

flashloan: 3245 rows x 11 cols


,dtype
time_bucket,str
asset,str
asset_symbol,str
flashloan_amount_raw,object
flashloan_premium_raw,object
flashloan_tx_count,int64
unique_flashloan_initiators,int64
no_open_debt_flashloan_tx_count,int64
stable_flashloan_tx_count,int64
variable_flashloan_tx_count,int64


,time_bucket,asset,asset_symbol,flashloan_amount_raw,flashloan_premium_raw,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,stable_flashloan_tx_count,variable_flashloan_tx_count,latest_flashloan_block
0,2025-11-01 00:00:00.000 UTC,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,215318475,107660,1,1,1,0,0,23700833
1,2025-11-01 00:00:00.000 UTC,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,4750869782538272168442,0,1,1,1,0,0,23702297
2,2025-11-01 00:00:00.000 UTC,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,174699718269860777,0,1,1,0,0,1,23701282
3,2025-11-01 00:00:00.000 UTC,0x6c3ea9036406852006290770bedfcaba0e23a0e8,PYUSD,1223755716915,0,4,1,0,0,4,23701970
4,2025-11-01 00:00:00.000 UTC,0x90d2af7d622ca3141efa4d8f1f24d86e5974cc8f,eUSDe,3160577892722259000000,1580288946361129500,1,1,1,0,0,23702389
5,2025-11-01 00:00:00.000 UTC,0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48,USDC,1549008553793,40382296,8,4,4,0,4,23702109
6,2025-11-01 00:00:00.000 UTC,0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2,WETH,180208047056986968317,19926263456270850,9,5,4,0,5,23702332
7,2025-11-01 00:00:00.000 UTC,0xc139190f447e929f090edeb554d95abb8b18ac1c,USDtb,57789515037665362521379,28894757518832681261,1,1,1,0,0,23702472
8,2025-11-01 00:00:00.000 UTC,0xcbb7c0000ab88b473b1f5afd9ef808440eed33bf,cbBTC,5502197,2752,1,1,1,0,0,23700806
9,2025-11-01 00:00:00.000 UTC,0xdac17f958d2ee523a2206206994597c13d831ec7,USDT,293243767124,50099775,4,3,2,0,2,23702238


In [154]:
TABLE = "query_7711298"

df_2_flashloan = tf.load_table(TABLE)
print(f"{TABLE}: {df_2_flashloan.shape[0]} rows x {df_2_flashloan.shape[1]} cols")
# display(df_2_flashloan.dtypes.to_frame("dtype"))
# display(df_2_flashloan.head(PREVIEW_ROWS))

# split the reference frame by its `metric` column
df_2_flashloan_amount  = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_amount_raw"].copy()
df_2_flashloan_premium = df_2_flashloan[df_2_flashloan["metric"] == "flashloan_premium_raw"].copy()

display(df_2_flashloan_amount.head(PREVIEW_ROWS))
display(df_2_flashloan_premium.head(PREVIEW_ROWS))

query_7711298: 95 rows x 5 cols


,asset,asset_symbol,metric,decimals,unit
0,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,flashloan_amount_raw,18,raw_token_amount
1,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,flashloan_amount_raw,18,raw_token_amount
2,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,flashloan_amount_raw,6,raw_token_amount
3,0x1f84a51296691320478c98b8d77f2bbd17d34350,PT-USDe-5FEB2026,flashloan_amount_raw,18,raw_token_amount
4,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,flashloan_amount_raw,18,raw_token_amount
5,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,flashloan_amount_raw,8,raw_token_amount
6,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,flashloan_amount_raw,6,raw_token_amount
7,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,flashloan_amount_raw,18,raw_token_amount
8,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,flashloan_amount_raw,18,raw_token_amount
9,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,flashloan_amount_raw,18,raw_token_amount


,asset,asset_symbol,metric,decimals,unit
43,0x111111111117dc0aa78b770fa6a738034120c302,1INCH,flashloan_premium_raw,18,raw_token_amount
44,0x18084fba666a33d37592fa2633fd49a74dd93a88,tBTC,flashloan_premium_raw,18,raw_token_amount
45,0x1abaea1f7c830bd89acc67ec4af516284b1bc33c,EURC,flashloan_premium_raw,6,raw_token_amount
46,0x1f84a51296691320478c98b8d77f2bbd17d34350,PT-USDe-5FEB2026,flashloan_premium_raw,18,raw_token_amount
47,0x1f9840a85d5af5bf1d1762f925bdaddc4201f984,UNI,flashloan_premium_raw,18,raw_token_amount
48,0x2260fac5e5542a773aa44fbcfedf7c193bc2c599,WBTC,flashloan_premium_raw,8,raw_token_amount
49,0x356b8d89c1e1239cbbb9de4815c39a1474d5ba7d,syrupUSDT,flashloan_premium_raw,6,raw_token_amount
50,0x40d16fc0246ad3160ccc09b8d0d3a2cd28ae6c2f,GHO,flashloan_premium_raw,18,raw_token_amount
51,0x4c9edd5852cd905f086c759e8383e09bff1e68b3,USDe,flashloan_premium_raw,18,raw_token_amount
52,0x514910771af9ca656af840dff83e8264ecf986ca,LINK,flashloan_premium_raw,18,raw_token_amount


In [159]:
flashloan_agg_cols = [
    "flashloan_tx_count",
    "unique_flashloan_initiators",
    "no_open_debt_flashloan_tx_count",
    "variable_flashloan_tx_count",
]

df_1_flashloan = tf.aggregate_by_time_bucket(df_0_flashloan, "time_bucket", flashloan_agg_cols, agg_func='sum')
df_1_flashloan

,time_bucket,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-11-01 00:00:00.000 UTC,32,20,16,16
1,2025-11-01 06:00:00.000 UTC,25,19,16,9
2,2025-11-01 12:00:00.000 UTC,40,25,17,23
3,2025-11-01 18:00:00.000 UTC,55,30,24,31
4,2025-11-02 00:00:00.000 UTC,30,13,8,22
...,...,...,...,...,...
363,2026-01-30 18:00:00.000 UTC,120,24,120,0
364,2026-01-31 00:00:00.000 UTC,22,9,22,0
365,2026-01-31 06:00:00.000 UTC,124,20,124,0
366,2026-01-31 12:00:00.000 UTC,514,36,514,0


In [162]:
df_2_flashloan_scaled = tf.scale_by_decimals(df_0_flashloan, df_2_flashloan_amount, drop_raw=True)
df_2_flashloan_scaled.drop(flashloan_agg_cols, axis=1, inplace=True)
df_2_flashloan_scaled.drop(["stable_flashloan_tx_count", "latest_flashloan_block"], axis=1, inplace=True)
df_2_flashloan_scaled_1 = tf.multiply_by_price(df_2_flashloan_scaled, df_1_borrow, ["flashloan_amount", "flashloan_premium"])
# df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')

stats_flashloan = [
    "flashloan_amount_value_usd",
    "flashloan_amount_value_eth",
    "flashloan_premium_value_usd",
    "flashloan_premium_value_eth",
]

In [166]:
df_2_flashloan_scaled_2 = tf.aggregate_by_time_bucket(df_2_flashloan_scaled_1, "time_bucket", stats_flashloan , agg_func='sum')
DF_flashloan_common = df_2_flashloan_scaled_2.merge(
    df_1_flashloan,
      on=["time_bucket"], how="left", suffixes=("_x", "_y"))

In [167]:
DF_flashloan_common

,time_bucket,flashloan_amount_value_usd,flashloan_amount_value_eth,flashloan_premium_value_usd,flashloan_premium_value_eth,flashloan_tx_count,unique_flashloan_initiators,no_open_debt_flashloan_tx_count,variable_flashloan_tx_count
0,2025-11-01 00:00:00.000 UTC,4.071241e+06,1056.130331,319.223135,0.082810,32,20,16,16
1,2025-11-01 06:00:00.000 UTC,3.378191e+06,872.889265,1226.461933,0.316905,25,19,16,9
2,2025-11-01 12:00:00.000 UTC,3.708287e+06,955.642607,95.912402,0.024717,40,25,17,23
3,2025-11-01 18:00:00.000 UTC,2.418851e+06,623.479460,440.544439,0.113554,55,30,24,31
4,2025-11-02 00:00:00.000 UTC,2.953926e+06,760.782920,434.396388,0.111879,30,13,8,22
...,...,...,...,...,...,...,...,...,...
363,2026-01-30 18:00:00.000 UTC,8.090294e+06,2990.313301,4014.400868,1.483792,120,24,120,0
364,2026-01-31 00:00:00.000 UTC,2.980801e+05,110.431780,149.040267,0.055216,22,9,22,0
365,2026-01-31 06:00:00.000 UTC,4.764459e+06,1790.737545,2367.473258,0.889823,124,20,124,0
366,2026-01-31 12:00:00.000 UTC,4.587787e+07,17964.775379,22110.499358,8.658010,514,36,514,0


In [64]:
TABLE = "user_account"

df_0_user_account = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_user_account.shape[0]} rows x {df_0_user_account.shape[1]} cols")
display(df_0_user_account.dtypes.to_frame("dtype"))
display(df_0_user_account.head(PREVIEW_ROWS))

user_account: 289 rows x 10 cols


,dtype
time_bucket,str
avg_total_collateral_base,float64
avg_total_debt_base,float64
avg_available_borrows_base,float64
avg_current_liquidation_threshold,float64
avg_ltv,float64
min_health_factor,object
max_health_factor,object
sampled_user_count,int64
account_data_call_count,int64


,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,min_health_factor,max_health_factor,sampled_user_count,account_data_call_count
0,2025-11-01 00:00:00.000 UTC,6.334549e+11,0.000000e+00,5.891131e+11,9500.000000,9300.000000,1157920892373161954235709850086879078532699846...,1157920892373161954235709850086879078532699846...,1,3
1,2025-11-01 06:00:00.000 UTC,1.857490e+16,9.730363e+15,4.564883e+15,7960.000000,7696.000000,1519550398599387828,1519550398599387828,1,1
2,2025-11-01 18:00:00.000 UTC,4.923167e+11,4.527922e+11,5.062364e+09,9500.000000,9300.000000,1032926139111913015,1032926139111913015,1,1
3,2025-11-02 00:00:00.000 UTC,6.195266e+15,3.248046e+15,1.519899e+15,8986.666667,8765.333333,1518215877808611369,1157920892373161954235709850086879078532699846...,2,3
4,2025-11-02 06:00:00.000 UTC,5.368513e+11,1.343379e+10,4.826504e+11,9260.000000,9050.000000,1575531656625301758,1157920892373161954235709850086879078532699846...,2,5
5,2025-11-02 18:00:00.000 UTC,4.916688e+11,4.522577e+11,4.994348e+09,9500.000000,9300.000000,1032786011671455792,1032786011671455792,1,1
6,2025-11-03 00:00:00.000 UTC,1.395521e+13,1.065452e+13,5.899184e+11,8900.000000,8675.000000,1041470613416050449,1504426874695587253,3,6
7,2025-11-03 06:00:00.000 UTC,3.761479e+12,2.269418e+12,7.192926e+11,8180.000000,7930.000000,1129265419073505776,1547794913062214560,3,5
8,2025-11-03 12:00:00.000 UTC,3.333276e+13,2.087862e+13,5.943480e+12,8268.625000,8002.937500,1099023921360700731,2060709910192696928,9,16
9,2025-11-03 18:00:00.000 UTC,7.117148e+13,5.297439e+13,2.917180e+12,8100.000000,7850.000000,1067221901564823484,1135277727315650182,2,3


In [74]:
stats_user = [
    df_0_user_account["sampled_user_count"].mean(), 
    df_0_user_account["sampled_user_count"].median(),
    # df_0_user_account["sampled_user_count"].std(),
    # df_0_user_account["sampled_user_count"].var(),
    df_0_user_account["account_data_call_count"].mean(),
    df_0_user_account["account_data_call_count"].median(),
#   df_0_user_account["account_data_call_count"].std(),
#   df_0_user_account["account_data_call_count"].var()
]

print(stats_user)

[np.float64(3.6366782006920415), np.float64(2.0), np.float64(8.487889273356402), np.float64(5.0)]


In [66]:
TABLE = "query_7711304"

df_0_user_account_agg = tf.load_table(TABLE)
print(f"{TABLE}: {df_0_user_account_agg.shape[0]} rows x {df_0_user_account_agg.shape[1]} cols")
display(df_0_user_account_agg.dtypes.to_frame("dtype"))
# display(df_0_user_account_agg.head(PREVIEW_ROWS))

df_0_user_account_agg_1 = df_0_user_account_agg.drop(df_0_user_account_agg.index[[0, 8]])
df_0_user_account_agg_1


query_7711304: 9 rows x 5 cols


,dtype
asset,float64
asset_symbol,float64
metric,str
decimals,int64
unit,str


,asset,asset_symbol,metric,decimals,unit
1,NaN,NaN,avg_available_borrows_base,8,usd_base_8dp
2,NaN,NaN,avg_current_liquidation_threshold,4,basis_points
3,NaN,NaN,avg_ltv,4,basis_points
4,NaN,NaN,avg_total_collateral_base,8,usd_base_8dp
5,NaN,NaN,avg_total_debt_base,8,usd_base_8dp
6,NaN,NaN,max_health_factor,18,wad
7,NaN,NaN,min_health_factor,18,wad


In [67]:
metric_cols = dict(zip(
    df_0_user_account_agg_1["metric"],
    df_0_user_account_agg_1["decimals"]
))

    
out = df_0_user_account.copy()

for col, dec in metric_cols.items():
    if col not in out.columns:
        continue  # skip metrics that aren't columns in this frame
    out[col] = out[col].astype("float64") / (10 ** int(dec))
df_0_user_account_agg_scaled = out
df_0_user_account_agg_scaled

,time_bucket,avg_total_collateral_base,avg_total_debt_base,avg_available_borrows_base,avg_current_liquidation_threshold,avg_ltv,min_health_factor,max_health_factor,sampled_user_count,account_data_call_count
0,2025-11-01 00:00:00.000 UTC,6.334549e+03,0.000000e+00,5.891131e+03,0.950000,0.930000,1.157921e+59,1.157921e+59,1,3
1,2025-11-01 06:00:00.000 UTC,1.857490e+08,9.730363e+07,4.564883e+07,0.796000,0.769600,1.519550e+00,1.519550e+00,1,1
2,2025-11-01 18:00:00.000 UTC,4.923167e+03,4.527922e+03,5.062364e+01,0.950000,0.930000,1.032926e+00,1.032926e+00,1,1
3,2025-11-02 00:00:00.000 UTC,6.195266e+07,3.248046e+07,1.519899e+07,0.898667,0.876533,1.518216e+00,1.157921e+59,2,3
4,2025-11-02 06:00:00.000 UTC,5.368513e+03,1.343379e+02,4.826504e+03,0.926000,0.905000,1.575532e+00,1.157921e+59,2,5
...,...,...,...,...,...,...,...,...,...,...
284,2026-01-30 18:00:00.000 UTC,1.428296e+05,1.059075e+05,8.671253e+03,0.816593,0.790486,1.074267e+00,1.486006e+00,6,14
285,2026-01-31 00:00:00.000 UTC,2.635171e+04,1.636404e+04,3.399747e+03,0.780000,0.750000,1.256066e+00,1.256070e+00,1,3
286,2026-01-31 06:00:00.000 UTC,3.435967e+06,2.146940e+06,5.521532e+05,0.808495,0.782245,1.027673e+00,2.872559e+00,9,20
287,2026-01-31 12:00:00.000 UTC,2.948305e+07,2.206159e+07,1.086046e+06,0.812614,0.784253,1.025000e+00,1.907177e+00,46,144


In [68]:
# borrow_repay                   <- query_result_data_7702164_20260612T204729Z.csv
# collateral_toggle              <- query_result_data_7711248_20260612T204801Z.csv
# flashloan                      <- query_result_data_7711227_20260612T204751Z.csv
# liquidation                    <- query_result_data_7711212_20260612T204746Z.csv
# oracle_price_usd_eth_weth_6h   <- query_result_data_7714873_20260615T092402Z.csv
# query_7711171                  <- query_result_data_7711171_20260612T204720Z.csv
# query_7711265                  <- query_result_data_7711265_20260612T204730Z.csv
# query_7711276                  <- query_result_data_7711276_20260612T204743Z.csv
# query_7711290                  <- query_result_data_7711290_20260612T204747Z.csv
# query_7711298                  <- query_result_data_7711298_20260612T204752Z.csv
# query_7711304                  <- query_result_data_7711304_20260612T204754Z.csv
# query_7711307                  <- query_result_data_7711307_20260612T204802Z.csv
# reserve_config                 <- query_result_data_7711190_20260612T204744Z.csv
# reserve_state_rates            <- query_result_data_7711042_20260612T204742Z.csv
# supply_withdraw                <- query_result_data_7702138_20260612T204719Z.csv
# user_account                   <- query_result_data_7711236_20260612T204753Z.csv

# # load one table (read-only) and display it
#  # <-- pick a label from Cell 2 (or pass a CSV path)

# df_0_borrow = tf.load_table(TABLE)
# print(f"{TABLE}: {df_0_borrow.shape[0]} rows x {df_0_borrow.shape[1]} cols")
# display(df_0_borrow.dtypes.to_frame("dtype"))
# display(df_0_borrow.head(PREVIEW_ROWS))

# df_0_borrow["last_borrow_rate"] = df_0_borrow["last_borrow_rate"].astype(float) / 10**27
# df_0_borrow

# #oracle_price_usd_eth_weth_6h =  7714873
# df_1_borrow = tf.load_table("oracle_price_usd_eth_weth_6h")
# print(f"{TABLE}: {df_1_borrow.shape[0]} rows x {df_1_borrow.shape[1]} cols")
# display(df_1_borrow.dtypes.to_frame("dtype"))
# display(df_1_borrow.head(PREVIEW_ROWS))

# df_0_borrow.drop(["net_debt_flow_raw","latest_borrow_block", "latest_repay_block"], axis=1, inplace=True)
# DF_common = df_0_borrow[["time_bucket", "asset", "asset_symbol", "last_borrow_rate"]].copy()
# df_0_borrow.drop(["last_borrow_rate"], axis=1, inplace=True)
# df_0_borrow

# # Cell — divide borrow_repay raw amount columns by 10**decimals -> real token units
# # decimals come from df_1_borrow (the oracle_price table's `decimals` column), per asset.
# df_2_borrow = tf.scale_by_decimals(df_0_borrow, df_1_borrow, drop_raw=True)

# print(f"raw cols scaled: {tf.raw_amount_columns(df_0_borrow)}")
# display(df_2_borrow.head(PREVIEW_ROWS))

# # the "_raw" amount columns became these scaled token-unit columns (drop_raw=True)
# amount_cols = [c[: -len(tf.RAW_SUFFIX)] for c in tf.raw_amount_columns(df_0_borrow)]
# print("amount columns:", amount_cols)

# # for each amount column: negative-value check, range check, deviation score
# # (plot=False -> values only, no plots)
# for col in amount_cols:
#     print(f"\n===== {col} =====")
#     neg = av.negative_value_check(df_2_borrow, col, plot=False)   # any negatives? how many?
#     # rng = av.range_check(df_2_borrow, col, plot=False)            # observed value range
#     # dev = av.deviation_score(df_2_borrow, col, plot=False)        # per-row score in [-1, 1]

#     print(f"negatives : {neg['n_negative']} of {neg['n_checked']} ({neg['negative_pct']}%)")
#     # print(f"range     : min={rng['observed_min']}  max={rng['observed_max']}")
#     # print(f"deviation : min={dev['score_min']:.3f}  max={dev['score_max']:.3f}  mean={dev['score_mean']:.3f}")





# df_2_borrow = tf.multiply_by_price(df_2_borrow, df_1_borrow, amount_cols)
# df_2_borrow



# borrow_agg_cols = [
#     "borrow_tx_count",
#     "repay_tx_count",
#     # "stable_borrow_tx_count",
#     "variable_borrow_tx_count",
#     "unique_borrowers",
#     "unique_repayers",
#     "borrow_amount_value_usd",
#     "borrow_amount_value_eth",
#     "repay_amount_value_usd",
#     "repay_amount_value_eth",
# ]

# df_3_borrow = tf.aggregate_by_time_bucket(df_2_borrow, "time_bucket", borrow_agg_cols, agg_func='sum')
# # df_3_borrow.drop(["stable_borrow_tx_count"], axis=1, inplace=True )

# df_3_borrow

# stats = pd.DataFrame({
#     "mean": df_3_borrow[borrow_agg_cols].mean(),
#     "median": df_3_borrow[borrow_agg_cols].median(),
#     "std": df_3_borrow[borrow_agg_cols].std(),
#     "variance": df_3_borrow[borrow_agg_cols].var(),
#     "cv": df_3_borrow[borrow_agg_cols].std() / df_3_borrow[borrow_agg_cols].mean(),
#     "p95": df_3_borrow[borrow_agg_cols].quantile(0.95),
#     "p99": df_3_borrow[borrow_agg_cols].quantile(0.99)
# })

# print(stats)

